In [1]:
import json 
from urllib.request import Request, urlopen 
 
TEAM_ID = "TEAM_38" 
API_KEY = "oc_Y012Xj-kjTJ9I6qLf9vhmiJKgPMvyDC5" 
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle" 
 
def api(payload): 
    req = Request(API_URL, data=json.dumps(payload).encode(), 
                  headers={"Content-Type":"application/json", "X-API-Key":API_KEY}) 
    with urlopen(req, timeout=30) as r: 
        return json.load(r) 
 
spec = api({"action":"spec", "team_id":TEAM_ID}) 
SPACE = spec["hyperparameters"] 
print("Model:", spec["model"]) 
for name, values in SPACE.items(): 
    print(name, ":", values) 

Model: SGDRegressor
eta0 : [0.0001, 0.001, 0.01, 0.05]
loss : ['squared_error', 'huber', 'epsilon_insensitive']
alpha : [1e-06, 3.727593720314938e-06, 1.389495494373136e-05, 5.1794746792312125e-05, 0.00019306977288832496, 0.0007196856730011514, 0.0026826957952797246, 0.01]
average : [False, True]
penalty : ['l2', 'l1', 'elasticnet']
learning_rate : ['constant', 'invscaling', 'adaptive', 'optimal']


In [5]:
# ============================================================
# SGDRegressor - Adaptive Hyperparameter Search
# Maximum possible calls = 2304
# ============================================================
import json 
from urllib.request import Request, urlopen 
 
TEAM_ID = "TEAM_38" 
API_KEY = "oc_Y012Xj-kjTJ9I6qLf9vhmiJKgPMvyDC5" 
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle" 
 
def api(payload): 
    req = Request(API_URL, data=json.dumps(payload).encode(), 
                  headers={"Content-Type":"application/json", "X-API-Key":API_KEY}) 
    with urlopen(req, timeout=30) as r: 
        return json.load(r) 
 
spec = api({"action":"spec", "team_id":TEAM_ID}) 
SPACE = spec["hyperparameters"] 
print("Model:", spec["model"]) 
for name, values in SPACE.items(): 
    print(name, ":", values) 
# ============================================================
# 3. ORACLE QUERY
# ============================================================

def oracle_query(params):

    result = api({
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    })

    return float(result["loss"])


# ============================================================
# 4. SEARCH SPACE
# ============================================================

eta0_values = [
    0.0001,
    0.001,
    0.01,
    0.05
]

loss_values = [
    "squared_error",
    "huber",
    "epsilon_insensitive"
]

alpha_values = [
    1e-06,
    3.727593720314938e-06,
    1.389495494373136e-05,
    5.1794746792312125e-05,
    0.00019306977288832496,
    0.0007196856730011514,
    0.0026826957952797246,
    0.01
]

average_values = [
    False,
    True
]

penalty_values = [
    "l2",
    "l1",
    "elasticnet"
]

learning_rate_values = [
    "constant",
    "invscaling",
    "adaptive",
    "optimal"
]


# ============================================================
# 5. SETTINGS
# ============================================================

# Complete search space:
#
# 4 x 3 x 8 x 2 x 3 x 4 = 2304
#
# This is the absolute maximum.

MAX_CALLS = 2304


# ============================================================
# 6. TRACKING
# ============================================================

call_count = 0

best_loss = float("inf")

best_params = None

tested = set()

history = []


# ============================================================
# 7. UNIQUE CONFIGURATION KEY
# ============================================================

def configuration_key(params):

    return (
        params["eta0"],
        params["loss"],
        params["alpha"],
        params["average"],
        params["penalty"],
        params["learning_rate"]
    )


# ============================================================
# 8. ORACLE EVALUATION
# ============================================================

def evaluate(params):

    global call_count
    global best_loss
    global best_params

    if call_count >= MAX_CALLS:
        return None

    key = configuration_key(params)

    # Don't query duplicates
    if key in tested:
        return None

    tested.add(key)

    # Oracle call
    current_loss = oracle_query(params)

    call_count += 1

    # Store history
    history.append({
        "call": call_count,
        "params": params.copy(),
        "loss": current_loss
    })

    # --------------------------------------------------------
    # Compare against BEST LOSS
    # --------------------------------------------------------

    if current_loss < best_loss:

        best_loss = current_loss

        best_params = params.copy()

        print("\n" + "=" * 70)
        print("NEW BEST")
        print("=" * 70)

        print("Call :", call_count)
        print("Loss :", current_loss)
        print("Params:", params)

    else:

        print(
            f"Call {call_count:4d} | "
            f"Loss = {current_loss:.8f} | "
            f"Best = {best_loss:.8f}"
        )

    return current_loss


# ============================================================
# 9. GENERATE ALL CONFIGURATIONS
# ============================================================
#
# We still use nested loops.
#
# But instead of immediately querying everything,
# we create the possible configurations first.
#

all_configs = []

for eta0 in eta0_values:

    for loss_function in loss_values:

        for alpha in alpha_values:

            for average in average_values:

                for penalty in penalty_values:

                    for learning_rate in learning_rate_values:

                        params = {
                            "eta0": eta0,
                            "loss": loss_function,
                            "alpha": alpha,
                            "average": average,
                            "penalty": penalty,
                            "learning_rate": learning_rate
                        }

                        all_configs.append(params)


print("\nTotal configurations:")
print(len(all_configs))


# ============================================================
# 10. INITIAL EXPLORATION
# ============================================================
#
# First test configurations that cover the parameter space.
#
# This gives us information about which parameter values
# are producing lower losses.
# ============================================================

initial_configs = [

    {
        "eta0": 0.001,
        "loss": "squared_error",
        "alpha": 0.00019306977288832496,
        "average": False,
        "penalty": "l2",
        "learning_rate": "constant"
    },

    {
        "eta0": 0.0001,
        "loss": "squared_error",
        "alpha": 1e-06,
        "average": False,
        "penalty": "elasticnet",
        "learning_rate": "adaptive"
    },

    {
        "eta0": 0.01,
        "loss": "huber",
        "alpha": 0.00019306977288832496,
        "average": True,
        "penalty": "l2",
        "learning_rate": "adaptive"
    },

    {
        "eta0": 0.05,
        "loss": "epsilon_insensitive",
        "alpha": 0.01,
        "average": True,
        "penalty": "elasticnet",
        "learning_rate": "optimal"
    }
]


print("\n")
print("=" * 70)
print("INITIAL EXPLORATION")
print("=" * 70)

for params in initial_configs:

    if call_count >= MAX_CALLS:
        break

    evaluate(params)


# ============================================================
# 11. SCORE PARAMETER VALUES
# ============================================================
#
# We calculate the average observed loss for every value.
#
# Lower average loss = more promising value.
# ============================================================

def calculate_value_scores():

    scores = {
        "eta0": {},
        "loss": {},
        "alpha": {},
        "average": {},
        "penalty": {},
        "learning_rate": {}
    }

    counts = {
        "eta0": {},
        "loss": {},
        "alpha": {},
        "average": {},
        "penalty": {},
        "learning_rate": {}
    }

    for item in history:

        params = item["params"]
        loss = item["loss"]

        for parameter in scores:

            value = params[parameter]

            if value not in scores[parameter]:

                scores[parameter][value] = 0
                counts[parameter][value] = 0

            scores[parameter][value] += loss
            counts[parameter][value] += 1

    # Convert totals into averages
    for parameter in scores:

        for value in scores[parameter]:

            scores[parameter][value] /= counts[parameter][value]

    return scores


# ============================================================
# 12. ADAPTIVE SEARCH
# ============================================================
#
# At each stage:
#
# 1. Calculate average loss for parameter values.
# 2. Identify promising values.
# 3. Search configurations containing those values.
# 4. Update the observations.
# 5. Repeat.
#
# We still keep the nested loops.
# ============================================================

while call_count < MAX_CALLS:

    scores = calculate_value_scores()

    # --------------------------------------------------------
    # If we don't have enough information yet, search
    # unexplored configurations.
    # --------------------------------------------------------

    candidate_configs = []

    for params in all_configs:

        key = configuration_key(params)

        if key in tested:
            continue

        # ----------------------------------------------------
        # Calculate a score for this configuration.
        #
        # Lower score = more promising.
        # ----------------------------------------------------

        score = 0

        known_parameters = 0

        for parameter in scores:

            value = params[parameter]

            if value in scores[parameter]:

                score += scores[parameter][value]

                known_parameters += 1

        # If no information is available, give it a neutral
        # priority so that it can still be explored.

        if known_parameters > 0:

            score = score / known_parameters

        else:

            score = float("inf")

        candidate_configs.append(
            (score, params)
        )


    # --------------------------------------------------------
    # Sort promising configurations first.
    # --------------------------------------------------------

    candidate_configs.sort(
        key=lambda x: x[0]
    )


    # --------------------------------------------------------
    # Search the best unexplored configurations.
    #
    # Use nested loops over a small batch.
    # --------------------------------------------------------

    batch_size = 20

    batch = candidate_configs[:batch_size]

    if not batch:
        break


    print("\n")
    print("=" * 70)
    print("ADAPTIVE SEARCH")
    print("=" * 70)

    for score, params in batch:

        if call_count >= MAX_CALLS:
            break

        evaluate(params)


    # --------------------------------------------------------
    # Progress information
    # --------------------------------------------------------

    print("\nCurrent progress:")
    print("Calls:", call_count)
    print("Best loss:", best_loss)


# ============================================================
# 13. FINAL RESULT
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULT")
print("=" * 70)

print("\nBest Hyperparameters:")

for parameter, value in best_params.items():

    print(
        f"{parameter}: {value}"
    )


print("\nBest Loss:")
print(best_loss)


print("\nTotal Oracle Calls:")
print(call_count)


print("\nUnique Configurations Tested:")
print(len(tested))


print("\nTotal Possible Configurations:")
print(len(all_configs))


print("\nPercentage of Search Space Tested:")

print(
    round(
        (call_count / len(all_configs)) * 100,
        2
    ),
    "%"
)


# ============================================================
# 14. COMPLETE HISTORY
# ============================================================

print("\n")
print("=" * 70)
print("ORACLE HISTORY")
print("=" * 70)

for result in history:

    print(
        f"Call {result['call']:4d} | "
        f"Loss = {result['loss']:.8f} | "
        f"Parameters = {result['params']}"
    )

Model: SGDRegressor
eta0 : [0.0001, 0.001, 0.01, 0.05]
loss : ['squared_error', 'huber', 'epsilon_insensitive']
alpha : [1e-06, 3.727593720314938e-06, 1.389495494373136e-05, 5.1794746792312125e-05, 0.00019306977288832496, 0.0007196856730011514, 0.0026826957952797246, 0.01]
average : [False, True]
penalty : ['l2', 'l1', 'elasticnet']
learning_rate : ['constant', 'invscaling', 'adaptive', 'optimal']

Total configurations:
2304


INITIAL EXPLORATION

NEW BEST
Call : 1
Loss : 14.2318777822747
Params: {'eta0': 0.001, 'loss': 'squared_error', 'alpha': 0.00019306977288832496, 'average': False, 'penalty': 'l2', 'learning_rate': 'constant'}

NEW BEST
Call : 2
Loss : 14.2283577478302
Params: {'eta0': 0.0001, 'loss': 'squared_error', 'alpha': 1e-06, 'average': False, 'penalty': 'elasticnet', 'learning_rate': 'adaptive'}
Call    3 | Loss = 26.70524856 | Best = 14.22835775
Call    4 | Loss = 25.28714461 | Best = 14.22835775


ADAPTIVE SEARCH
Call    5 | Loss = 38.70484454 | Best = 14.22835775
Call 

TimeoutError: The read operation timed out